# Clase 11 — Carga Incremental: dos patrones sobre datos SAP
## Modulo 7 - SAP + Databricks Integration Real

**Caso de estudio:** Grupo Argos — cierre mensual de ventas y retorno de resultados ML

---

### Los dos patrones

| Patron | Direccion | Tecnologia | Cuando usarlo |
|--------|-----------|------------|---------------|
| **MERGE Strategy** | SAP -> Databricks Gold | Delta MERGE + CASE WHEN | Acumulacion mensual |
| **CDF de Retorno** | Databricks Gold -> SAP | Change Data Feed | ML results -> Datasphere |

### Por que BDC no habilita CDF hacia SAP Databricks?
> BDC garantiza que el Delta Share siempre tiene el snapshot mas reciente de S/4HANA.
> La incrementalidad ocurre DENTRO de BDC via CDC interno (APRS).
> El receptor lee el snapshot actualizado — no necesita el historial de deltas.

## 0. Setup

In [0]:
CATALOG = 'laboratory_sap_dev'
SCHEMA  = 'sap_course'
spark.sql(f'USE CATALOG {CATALOG}')
spark.sql(f'USE SCHEMA {SCHEMA}')
print(f'OK: {CATALOG}.{SCHEMA}')

for t in ['vbak_silver','gold_sales_kpis','gold_customer_360']:
    try:
        print(f'OK {t}: {spark.table(t).count():,} filas')
    except Exception as e:
        print(f'ERR {t}: {e}')

---
## Ejercicio 1 — MERGE Strategy: nuevas ordenes SAP -> Gold KPIs

### Por que no hay CDF de entrada desde BDC?
BDC garantiza que el share siempre tiene el estado mas reciente. No necesitas los deltas de entrada.
Lo que si necesitas es una estrategia para ACUMULAR esos nuevos datos en tu Gold existente.

```
SIN MERGE strategy:
  Llegan 5 ordenes nuevas -> borras gold_sales_kpis completo
  -> Recalculas todo desde cero (N millones de filas) X

CON MERGE strategy:
  Llegan 5 ordenes nuevas -> MERGE acumula solo esas 5
  -> Si el mes existe: UPDATE sumando nuevos valores
  -> Si el mes es nuevo: INSERT solo ese registro OK
```

In [0]:
# Agregar columna de control de actualización
spark.sql('''
    ALTER TABLE vbak_silver 
    ADD COLUMN updated_at TIMESTAMP
''')

# Setear fecha pasada para todos los registros históricos
spark.sql('''
    UPDATE vbak_silver 
    SET updated_at = TIMESTAMP('2024-12-31 00:00:00')
    WHERE updated_at IS NULL
''')

print('OK: columna updated_at agregada')
spark.sql('SELECT updated_at, COUNT(*) FROM vbak_silver GROUP BY updated_at').show()

### Paso 1.1 — Estado actual del Gold

In [0]:
spark.sql('''
    SELECT year, month, sales_org, currency,
           num_orders,
           ROUND(total_revenue, 2)    AS total_revenue,
           ROUND(avg_order_value, 2)  AS avg_order_value
    FROM gold_sales_kpis
    ORDER BY year DESC, month DESC, sales_org
    LIMIT 15
''').show(truncate=False)

### Paso 1.2 — Simular nuevas ordenes llegando del share BDC

In [0]:
from pyspark.sql.functions import to_date, to_timestamp, col

nuevas_ordenes = spark.createDataFrame([
    ('9000000001', '20250601', '0000001000', '1000', 15200000.0, 'COP', 'TA', '2025-06-01 08:00:00'),
    ('9000000002', '20250605', '0000001001', '1000', 18500000.0, 'COP', 'TA', '2025-06-01 08:00:00'),
    ('9000000003', '20250610', '0000001002', '1000', 22100000.0, 'COP', 'TA', '2025-06-01 08:00:00'),
    ('9000000004', '20250603', '0000002000', '2000',    52000.0, 'USD', 'TA', '2025-06-01 08:00:00'),
    ('9000000005', '20250612', '0000002001', '2000',    67500.0, 'USD', 'TA', '2025-06-01 08:00:00'),
    ('9000000006', '20250601', '0000003000', '3000', 31400000.0, 'COP', 'TA', '2025-06-01 08:00:00'),
], ['VBELN','ERDAT','KUNNR','VKORG','NETWR','WAERK','AUART','updated_at'])

nuevas_ordenes = (nuevas_ordenes
    .withColumn('ERDAT',      to_date(col('ERDAT'), 'yyyyMMdd'))
    .withColumn('updated_at', to_timestamp(col('updated_at')))
)

nuevas_ordenes.write.format('delta').mode('append').saveAsTable('vbak_silver')

print(f'OK: {nuevas_ordenes.count()} ordenes insertadas')
spark.sql('''
    SELECT updated_at, COUNT(*) AS filas
    FROM vbak_silver
    GROUP BY updated_at
    ORDER BY updated_at
''').show()

### Paso 1.3 — Insertar en Silver via MERGE

In [0]:
# spark.sql('''
#     MERGE INTO vbak_silver AS target
#     USING nuevas_ordenes_sap AS source
#     ON target.VBELN = source.VBELN
#     WHEN NOT MATCHED THEN
#         INSERT (VBELN, ERDAT, KUNNR, VKORG, NETWR, WAERK, AUART)
#         VALUES (
#             source.VBELN,
#             TO_DATE(source.ERDAT, "yyyyMMdd"),
#             source.KUNNR,
#             source.VKORG,
#             source.NETWR,
#             source.WAERK,
#             source.AUART
#         )
# ''')
# print('OK: ordenes insertadas en Silver')

### Paso 1.4 — Calcular KPIs del periodo nuevo

### Paso 1.5 — MERGE Strategy con CASE WHEN al Gold

In [0]:
spark.sql('''
    MERGE INTO gold_sales_kpis AS g
    USING (
        SELECT
            YEAR(ERDAT)   AS year,
            MONTH(ERDAT)  AS month,
            VKORG         AS sales_org,
            WAERK         AS currency,
            COUNT(*)      AS num_orders_nuevas,
            SUM(NETWR)    AS revenue_nuevo,
            AVG(NETWR)    AS avg_nuevo
        FROM vbak_silver
        WHERE updated_at = (
            SELECT MAX(updated_at) FROM vbak_silver
        )
        GROUP BY 1, 2, 3, 4
    ) AS n
    ON  g.year      = n.year
    AND g.month     = n.month
    AND g.sales_org = n.sales_org
    AND g.currency  = n.currency

    WHEN MATCHED THEN UPDATE SET
        g.num_orders      = g.num_orders + n.num_orders_nuevas,
        g.total_revenue   = g.total_revenue + n.revenue_nuevo,
        g.avg_order_value = CASE
            WHEN (g.num_orders + n.num_orders_nuevas) > 0
            THEN (g.total_revenue + n.revenue_nuevo)
                 / (g.num_orders + n.num_orders_nuevas)
            ELSE 0
        END

    WHEN NOT MATCHED THEN INSERT (
        year, month, sales_org, currency,
        num_orders, total_revenue, avg_order_value
    ) VALUES (
        n.year, n.month, n.sales_org, n.currency,
        n.num_orders_nuevas, n.revenue_nuevo, n.avg_nuevo
    )
''')

print('OK: gold_sales_kpis actualizado')
spark.sql('''
    SELECT year, month, sales_org, currency,
           num_orders,
           ROUND(total_revenue, 2)   AS total_revenue,
           ROUND(avg_order_value, 2) AS avg_order_value
    FROM gold_sales_kpis
    WHERE year = 2025
    ORDER BY month, sales_org
''').show()

### Paso 1.6 — Validar resultado del MERGE

In [0]:
from pyspark.sql.functions import to_date, to_timestamp, col

# Segundo lote — mismo mes junio 2025, debe hacer UPDATE acumulando
nuevas_ordenes_2 = spark.createDataFrame([
    ('9000000007', '20250620', '0000001003', '1000',  9800000.0, 'COP', 'TA', '2025-06-15 08:00:00'),
    ('9000000008', '20250622', '0000002002', '2000',    38000.0, 'USD', 'TA', '2025-06-15 08:00:00'),
], ['VBELN','ERDAT','KUNNR','VKORG','NETWR','WAERK','AUART','updated_at'])

nuevas_ordenes_2 = (nuevas_ordenes_2
    .withColumn('ERDAT',      to_date(col('ERDAT'), 'yyyyMMdd'))
    .withColumn('updated_at', to_timestamp(col('updated_at')))
)

nuevas_ordenes_2.write.format('delta').mode('append').saveAsTable('vbak_silver')
print(f'OK: {nuevas_ordenes_2.count()} ordenes nuevas insertadas en Silver')

# Verificar los dos lotes
spark.sql('''
    SELECT updated_at, COUNT(*) AS filas
    FROM vbak_silver
    GROUP BY updated_at
    ORDER BY updated_at
''').show()

### Por que CASE WHEN en avg_order_value?

```
SIN CASE WHEN -- INCORRECTO:
  Historico: 10 ordenes, revenue=100M -> avg=10M
  Nuevas:     3 ordenes, revenue=45M
  UPDATE avg = 45M/3 = 15M  <- solo promedio de las 3 nuevas, pierde historico X

CON CASE WHEN -- CORRECTO:
  avg = (100M + 45M) / (10 + 3) = 11.15M  <- promedio ponderado del periodo X
```

**Conexion con BDC:** Este MERGE es lo que haces despues de leer el snapshot
del share de BDC. El share ya tiene todas las ordenes actualizadas —
tu decides como acumularlas eficientemente en tu Gold.

In [0]:
%sql
-- VE al trial y ejecuta

-- SELECT 
--     year, month, sales_org, currency,
--     num_orders,
--     ROUND(total_revenue, 2)   AS total_revenue,
--     ROUND(avg_order_value, 2) AS avg_order_value
-- FROM sap_gold_shared.sales.monthly_kpis
-- WHERE year = 2025
-- ORDER BY month, sales_org

---
## Ejercicio 2 — CDF de Retorno: Gold -> OpenSharing -> Receptor

### Por que la vuelta SI usa CDF?

```
SIN CDF (receptor descarga todo):
  gold_sales_kpis tiene 1,000 filas
  Actualizas 3 filas de junio 2025
  El receptor descarga 1,000 filas otra vez  X

CON CDF (receptor descarga solo cambios):
  gold_sales_kpis tiene 1,000 filas
  Actualizas 3 filas de junio 2025
  El receptor descarga solo 3 filas  OK
```

**Equivalente en BDC:** el `enableChangeDataFeed=true` del Company_Clustering.ipynb.
Cuando escribe company_code_clusters, Datasphere descarga solo los clusters
que cambiaron — no los 500 Company Codes completos.

### Paso 2.1 — Habilitar CDF en gold_sales_kpis

In [0]:
# Celda 1 — Habilitar CDF en gold_customer_360
spark.sql('''
    ALTER TABLE gold_customer_360
    SET TBLPROPERTIES (
        'delta.enableChangeDataFeed'  = 'true',
        'delta.enableDeletionVectors' = 'false'
    )
''')
print('OK: CDF habilitado en gold_customer_360')

### Paso 2.2 — Capturar version actual antes del proximo cambio

In [0]:
# Celda 2 — Capturar versión
hist = spark.sql('DESCRIBE HISTORY gold_customer_360 LIMIT 1').collect()
version_antes = hist[0]['version']
print(f'Version actual: {version_antes}')
print(f'El receptor leerá CDF desde version: {version_antes + 1}')

### Paso 2.3 —Simular actualización de scores de clientes

In [0]:
# Celda 3 — Simular actualización de scores de clientes
# En producción: el modelo de churn corrió y actualizó estos clientes
spark.sql('''
    UPDATE gold_customer_360
    SET customer_tier = 'PLATINUM'
    WHERE total_revenue > (
        SELECT PERCENTILE(total_revenue, 0.90)
        FROM gold_customer_360
    )
''')
print('OK: tier actualizado para clientes top 10% de revenue')
print('OK: julio 2025 insertado')

### Paso 2.4 — Leer SOLO los cambios via CDF

In [0]:
# Celda 4 — Leer SOLO los cambios via CDF
cambios_cdf = (spark.read.format('delta')
    .option('readChangeFeed', 'true')
    .option('startingVersion', version_antes + 1)
    .table('gold_customer_360'))

total = spark.table('gold_customer_360').count()
cambios = cambios_cdf.count()

print(f'Total clientes en gold_customer_360:    {total:,}')
print(f'Clientes que viajan al receptor (CDF):  {cambios}')
print()
cambios_cdf.select(
    'customer_id','customer_name','customer_tier',
    'total_revenue','_change_type','_commit_version'
).show(truncate=False)

### Paso 2.5 — Clasificar tipos de cambio CDF

In [0]:
# Recrear el DataFrame CDF en la misma celda para evitar el scope issue
cambios_cdf = (spark.read.format('delta')
    .option('readChangeFeed', 'true')
    .option('startingVersion', version_antes + 1)
    .table('gold_customer_360'))

# Registrar como vista SQL — más estable para columnas de sistema
cambios_cdf.createOrReplaceTempView('cambios_cdf_view')

total = spark.table('gold_customer_360').count()

print('Tipos de cambios detectados:')
spark.sql('''
    SELECT _change_type, COUNT(*) AS filas
    FROM cambios_cdf_view
    GROUP BY _change_type
''').show()

print('Lo que recibe SAP Datasphere:')
spark.sql('''
    SELECT customer_id, customer_name,
           customer_tier, total_revenue, _change_type
    FROM cambios_cdf_view
    WHERE _change_type IN ('insert', 'update_postimage')
''').show(truncate=False)

total_receptor = spark.sql('''
    SELECT COUNT(*) FROM cambios_cdf_view
    WHERE _change_type IN ('insert', 'update_postimage')
''').collect()[0][0]

print(f'Clientes que Datasphere descarga: {total_receptor}')
print(f'En vez de los {total:,} clientes completos')

### Paso 2.6 — Verificar desde el receptor (ejecutar en el trial)

In [0]:
# EJECUTAR DESDE EL TRIAL (workspace receptor)
# El share ya refleja los cambios automaticamente via OpenSharing

# SELECT
#     customer_id,
#     customer_name,
#     customer_tier,
#     ROUND(total_revenue, 2) AS total_revenue
# FROM sap_gold_shared.customers.customer_360
# WHERE customer_tier = 'PLATINUM'
# ORDER BY total_revenue DESC
print('Ejecutar el bloque comentado desde el trial para verificar el share')

### Por que enableDeletionVectors = false?

```
Delta Lake moderno usa Deletion Vectors para marcar borrados
(archivos .dv en el storage — muy eficientes)

SAP Datasphere NO puede leer Deletion Vectors
-> enableDeletionVectors = false
-> Los borrados se representan como filas 'delete' en el CDF log
-> Datasphere SI puede procesarlas

Trade-off:
  false = escrituras ligeramente mas lentas, compatible con BDC
  true  = mas rapido pero incompatible con BDC/Datasphere

Regla: si la tabla va a ser compartida con BDC -> siempre false
```

---
## Comparacion final — Los dos patrones en Grupo Argos

| | Ejercicio 1 — MERGE Strategy | Ejercicio 2 — CDF Retorno |
|--|-------------------------------|---------------------------|
| **Direccion** | SAP BDC -> Databricks Gold | Databricks Gold -> SAP Datasphere |
| **CDF de entrada?** | NO — BDC envia snapshot | SI — CDF=true en la tabla |
| **Por que?** | El share ya esta actualizado | Datasphere descarga solo cambios |
| **Patron** | MERGE + CASE WHEN | readChangeFeed=true |
| **En BDC** | Leer share -> MERGE propio | company_code_clusters -> Datasphere |
| **En el curso** | vbak_silver -> gold_sales_kpis | gold_sales_kpis -> trial ecotickets |

```
La autopista bidireccional SAP:

S/4HANA -CDC-> BDC Foundation -Delta Sharing-> SAP Databricks
                               (snapshot fresco)      |
                                               MERGE Strategy
                                                      |
                                              Gold Tables
                                                      |
                                              CDF = true
                                                      |
                        SAP Datasphere <-Delta Sharing-+
                        (solo cambios via CDF)

IDA:    snapshot completo — la incrementalidad la hace BDC internamente
VUELTA: solo cambios — CDF lleva al receptor solo lo que modifico
```

---
## Preguntas de reflexion

1. **Por que CASE WHEN en avg_order_value y no simplemente avg = n.avg_nuevo?**

2. **Si BDC no habilita CDF hacia SAP Databricks, como sabe el clustering notebook
   que Company Codes son nuevos desde la ultima ejecucion?**
   (Pista: no lo sabe — siempre recalcula todo. Cuando es aceptable?)

3. **enableDeletionVectors=false sacrifica rendimiento para ganar compatibilidad.
   En que escenario ese trade-off NO valdria la pena?**

4. **En el proyecto integrador tu tabla Gold tiene CDF=true. Si el companero
   hace SELECT sin filtro de version, recibe todos los datos o solo los cambios?**
   (Pista: SELECT normal siempre lee el estado actual, no el CDF log)